# BaseAL Embedding Visualisation Example

This notebook demonstrates how to use the updated API:

```python
visualize_embeddings(dataset_path, idxes, with_label)
```

The meaning of `idxes` is:
- `idxes` is a 0-based array of sample indices **after matching available embeddings**
- In other words, rows in `labels.csv` without a corresponding embedding file are removed first, and the remaining samples are then re-indexed
- `idxes=None` means all matchable samples are used
- `idxes` must be a 1D integer array with no duplicates and no out-of-range values

Dataset root requirements:
- Must contain `labels.csv`
- Must contain `embeddings/<model_name>/*.npy`
- Files are matched using `Path(filename).stem + f"_{model_name}.npy"`


In [ ]:
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path("/scratch/asignal/shiqi/Project/BaseAL")  # Change this to your actual project root path

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.utils.visualization import visualize_embeddings


## What Is In the Returned Object

`visualize_embeddings(...)` returns a `VisualizationResult`. Common fields include:
- `viz.idxes`: the matched-sample indices actually used in this run
- `viz.annotations`: the corresponding annotation table
- `viz.embeddings`: the original embedding matrix
- `viz.coordinates`: the 2D dimensionality-reduction result
- `viz.labels`: a 1D integer array for single-label data, or a binary matrix for multi-label data
- `viz.label_names`: human-readable label strings for each sample
- `viz.label_indices_for_color`: the primary label index used for coloring each sample
- `viz.class_names`: the list of class names for the full dataset
- `viz.figure`, `viz.axes`: matplotlib objects


In [ ]:
# Single-label example: ESC10
multi_class_ds_path = PROJECT_ROOT / "ESC10_BASEAL"  # Change this to your actual dataset path
multi_class_ds_idxes = np.arange(200, dtype=np.int64)

viz_single = visualize_embeddings(
    dataset_path=multi_class_ds_path,
    idxes=multi_class_ds_idxes,
    with_label=True,
    max_reference_samples=500,
)

print("Used idxes shape:", viz_single.idxes.shape)
print("First 10 idxes:", viz_single.idxes[:10])
print("Reduction steps:", viz_single.reduction_steps)
print("Coordinates shape:", viz_single.coordinates.shape)
print("Embeddings shape:", viz_single.embeddings.shape)
print("Labels shape:", viz_single.labels.shape)
print("Is multilabel:", viz_single.is_multilabel)
print("Classes:", viz_single.class_names)

In [ ]:
viz_single.annotations[["matched_index", "annotation_index", "filename", "label"]].head(10)


## How Multi-Label Samples Are Handled

For samples whose `label` column looks like `"a;b;c"`, the utility will:
- Convert each sample into a binary label vector stored in `viz.labels`
- Join all active labels into a readable string stored in `viz.label_names`
- Generate an additional primary-label index `viz.label_indices_for_color` for scatter coloring

Note: the primary label is the **first active class** in the sample's binary vector, meaning the active class with the smallest index after `viz.class_names` ordering. This matches the current BaseAL API coloring semantics.

Default coloring logic:
- Only consider classes that actually appear in the current plot
- Assign colors to those present classes evenly on a fixed color wheel
- `no_call` / `noise` / `negative` are treated like normal classes by default and are not automatically colored gray


In [ ]:
# Multi-label example: ATBFL
# You can also replace this with HSN:
import os
multi_label_ds_path = Path(os.getenv("LOCAL_SCRATCH")) / "ATBFL_BASEAL"  # Change this to your actual dataset path
multi_label_ds_idxes = np.random.choice(500000, size=200, replace=False)

viz_multi = visualize_embeddings(
    dataset_path=multi_label_ds_path,
    idxes=multi_label_ds_idxes,
    with_label=True,
)

print("Used idxes:", viz_multi.idxes)
print("Reduction steps:", viz_multi.reduction_steps)
print("Coordinates shape:", viz_multi.coordinates.shape)
print("Embeddings shape:", viz_multi.embeddings.shape)
print("Labels shape:", viz_multi.labels.shape)
print("Is multilabel:", viz_multi.is_multilabel)
print("First 10 class names:", viz_multi.class_names[:10])

In [ ]:
import pandas as pd

inspect_df = viz_multi.annotations[["matched_index", "annotation_index", "filename", "label"]].copy()
inspect_df["rendered_label_names"] = viz_multi.label_names
inspect_df["color_label_index"] = viz_multi.label_indices_for_color
inspect_df["color_label_name"] = [
    viz_multi.class_names[idx] for idx in viz_multi.label_indices_for_color
]

inspect_df


## If You Want to Force Negative Samples to Gray

The example below redraws `no_call` / `noise` / `negative` in gray, while keeping the default behavior for other classes: evenly assigning colors only across classes that actually appear in the current plot.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

negative_labels = {"no_call", "noise", "negative"}
present_indices = np.unique(viz_multi.label_indices_for_color)

if len(present_indices) == 1:
    hues = np.array([2.0 / 3.0], dtype=np.float32)
else:
    hues = np.linspace(0.0, 1.0, len(present_indices), endpoint=False, dtype=np.float32)

rgb = hsv_to_rgb(
    np.column_stack([
        hues,
        np.full(len(present_indices), 0.8, dtype=np.float32),
        np.full(len(present_indices), 0.9, dtype=np.float32),
    ])
)
class_color_lookup = {
    int(class_idx): rgb[pos] for pos, class_idx in enumerate(present_indices)
}

point_colors = []
for class_idx in viz_multi.label_indices_for_color:
    class_name = viz_multi.class_names[class_idx]
    if class_name in negative_labels:
        point_colors.append("#9E9E9E")
    else:
        point_colors.append(class_color_lookup[int(class_idx)])

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
ax.scatter(
    viz_multi.coordinates[:, 0],
    viz_multi.coordinates[:, 1],
    c=point_colors,
    s=20,
    alpha=0.8,
    linewidths=0,
)
ax.set_title("ATBFL with negative/no_call samples forced to grey")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.grid(alpha=0.2)
plt.show()
